In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import time
import numpy as np
import os
import pandas as pd
pd.options.display.max_columns = 100
pd.options.display.max_rows = 130

In [11]:
from utils_data import (
    load_raw_data, check_dups)
from settings_lt_shorts import (
    audio_settings, create_directories, load_data_settings, lt_shorts_categories, load_video_configs
)
from utils_lt_shorts import (
    stitch_audios, draw_lt_vocab_list_whole_image, create_video_with_concat_images
)

# Get data

In [12]:
truly_load_data = True
if truly_load_data:
    df_all_vocab = load_raw_data()
    df_all_vocab.to_csv('static/latest_data.csv', index=False)
else:
    df_all_vocab = pd.read_csv('static/latest_data.csv')
    print('!!!!!!!! WARNING: not truly loading data !!!!!!!!')

df_dups = check_dups(df_all_vocab)
print(df_all_vocab.shape)
print(f'# duplicate vocab: {len(df_dups)}')
df_all_vocab.head(3)

(7488, 38)
# duplicate vocab: 0


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,1,房贷,fáng dài,mortgage,word,1.0,life,NaN,Finance & Economy,NaN,NaN,NONE,1.0,1.0,2.0,1.0,房子,house,贷款,loan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他每个月都要还房贷,Ta měi gè yuè dōu yào huán fángdài,He has to pay his mortgage every month,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,2,白天,bái tiān,daytime,word,2.0,time,NaN,Time,NaN,NaN,1,2.0,1.0,1.0,1.0,白,white,天,day,NaN,NaN,NaN,NaN,NaN,NaN,NaN,白天很热晚上比较凉快,Báitiān hěn rè wǎnshàng bǐjiào liángkuai,It is hot in the daytime and cooler at night,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,3,组成,zǔ chéng,to form;make up,word,3.0,general,NaN,Language & Expression,NaN,NaN,2,5.0,5.0,5.0,3.0,组,set,成,become,NaN,NaN,NaN,NaN,NaN,NaN,NaN,水是由氢和氧组成的,Shuǐ shì yóu qīng hé yǎng zǔchéng de,Water is made up of hydrogen and oxygen,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


# Get settings

In [ ]:
# '狗品种', '水生动物', '宠物'
current_category = ''
current_category_dict = lt_shorts_categories[current_category]
data_settings = load_data_settings(current_category_dict)
create_directories(data_settings)
data_settings

{'chinese': '爬行动物',
 'pinyin': 'pá xíng dòng wù',
 'english': 'reptiles',
 'vocab_list': ['壁虎', '变色龙', '响尾蛇'],
 'output_path': 'output/lt_shorts/爬行动物',
 'output_path_audio': 'output/lt_shorts/爬行动物/audio_files',
 'output_path_images': 'output/lt_shorts/爬行动物/images'}

In [14]:
# get list of all need for audio
df_for_audio = df_all_vocab[df_all_vocab['chinese'].isin([x for y in data_settings['vocab_list'] for x in y])].reset_index(drop=True)
all_audios = [data_settings['chinese']] + df_for_audio['chinese'].tolist() + df_for_audio['sentence'].tolist() + df_for_audio['word1'].tolist() + df_for_audio['word2'].tolist() + df_for_audio['word3'].tolist() + df_for_audio['word4'].tolist()
all_audios = [x for x in list(set(all_audios)) if not pd.isna(x)]
print(all_audios)

all_audios_english = df_for_audio['english'].tolist()
all_audios_english

['爬行动物', '蛇', '蛇通常生活在野外。']


['snake']

In [15]:
if data_settings['vocab_list'][0].__class__ == list:
    n_parts = len(data_settings['vocab_list'])
    data_settings['n_parts'] = n_parts
else:
    n_parts = 1

if n_parts > 1:
    for i in range(n_parts):
        df_vocab_list = df_all_vocab[df_all_vocab['chinese'].isin(data_settings['vocab_list'][i])].reset_index(drop=True)
        data_settings['current_part'] = i + 1
        print(f'Processing part {i+1} of {n_parts}')
        df_durations = stitch_audios(audio_settings, data_settings, None, df_vocab_list, part_number=i+1)
        video_configs = load_video_configs()
        final_img_file_path = draw_lt_vocab_list_whole_image(video_configs, data_settings, df_vocab_list)
        create_video_with_concat_images(df_durations, df_vocab_list, data_settings)
else:
    df_vocab_list = df_all_vocab[df_all_vocab['chinese'].isin(data_settings['vocab_list'])].reset_index(drop=True)
    df_durations = stitch_audios(audio_settings, data_settings, None, df_vocab_list)
    video_configs = load_video_configs()
    final_img_file_path = draw_lt_vocab_list_whole_image(video_configs, data_settings, df_vocab_list)
    create_video_with_concat_images(df_durations, df_vocab_list, data_settings)


Audio duration: 33.4s
Adding clip: output/lt_shorts/爬行动物/images/title_only.png for duration 2.3s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_0_component_only.png for duration 3.0s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_0_full.png for duration 3.4s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_0_sentence.png for duration 2.4s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_1_component_only.png for duration 4.9s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_1_full.png for duration 3.7s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_1_sentence.png for duration 2.5s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_2_component_only.png for duration 5.0s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_2_full.png for duration 3.9s
Adding clip: output/lt_shorts/爬行动物/images/vocab_word_2_sentence.png for duration 2.2s
Final audio duration: 33.410s
Final video duration before audio set: 33.360s
MoviePy - Building video output/lt_shorts/爬行动物

MoviePy - Done.
MoviePy - Writing video output/lt_shorts/爬行动物/爬行动物_video.mp4



MoviePy - Done !
MoviePy - video ready output/lt_shorts/爬行动物/爬行动物_video.mp4


In [16]:
# from moviepy import ImageClip
# img_show = ImageClip(final_img_file_path, duration=1).with_start(0)
# img_show.display_in_notebook()